In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/telecom_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)


In [9]:
# Q16: Service combination with lowest churn
combos = df.groupby(['InternetService', 'TechSupport'])['Churn'].apply(
    lambda x: round((x == 'Yes').sum() / len(x) * 100, 2)
).reset_index()
combos.columns = ['InternetService', 'TechSupport', 'churn_rate']
combos.sort_values('churn_rate')

,InternetService,TechSupport,churn_rate
4,No,No internet service,7.40
1,DSL,Yes,9.68
3,Fiber optic,Yes,22.63
0,DSL,No,27.76
2,Fiber optic,No,49.37


In [ ]:
# Q17: Services most strongly associated with churn reduction
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

results = []
for service in service_cols:
    # churn rate among customers WHO HAVE this service
    # df[df[service] == 'Yes'] filters to only rows where the customer subscribed
    churn_with = round((df[df[service] == 'Yes']['Churn'] == 'Yes').sum() /
                    len(df[df[service] == 'Yes']) * 100, 2)

    # churn rate among customers who DON'T have this service
    churn_without = round((df[df[service] == 'No']['Churn'] == 'Yes').sum() /
                    len(df[df[service] == 'No']) * 100, 2)

    # positive number = having this service reduces churn (churn_without > churn_with)
    # higher reduction = more protective against leaving
    reduction = round(churn_without - churn_with, 2)

    results.append({
        'service'        : service,
        'churn_with'     : churn_with,
        'churn_without'  : churn_without,
        'churn_reduction': reduction
    })

pd.DataFrame(results).sort_values('churn_reduction', ascending=False)

In [4]:
# Q18: Build a Service Adoption Score and analyze its relationship with churn
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

df['adoption_score'] = df[service_cols].apply(
    lambda row: (row == 'Yes').sum(), axis=1
)

df.groupby('adoption_score')['Churn'].apply(
    lambda x: round((x == 'Yes').sum() / len(x) * 100, 2)
).reset_index().rename(columns={'Churn': 'churn_rate'})

,adoption_score,churn_rate
0,0,21.41
1,1,45.76
2,2,35.82
3,3,27.37
4,4,22.30
5,5,12.43
6,6,5.28


In [5]:
# Q19: Fiber optic, no tech support, no online security — churn rate
high_risk = df[
    (df['InternetService'] == 'Fiber optic') &
    (df['TechSupport'] == 'No') &
    (df['OnlineSecurity'] == 'No')
]

print(f'Customers matching criteria: {len(high_risk)}')
print(f'Churned: {(high_risk["Churn"] == "Yes").sum()}')
print(f'Churn rate: {round((high_risk["Churn"] == "Yes").sum() / len(high_risk) * 100, 2)}%')

Customers matching criteria: 1765
Churned: 971
Churn rate: 55.01%


In [ ]:
# Q20: Most profitable service bundle
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

# for each customer, build a string listing all services they subscribe to, joined by '+'
# a customer with OnlineSecurity + TechSupport → 'OnlineSecurity+TechSupport'
# a customer with nothing → '' (empty string)
df['bundle'] = df[service_cols].apply(
    lambda row: '+'.join([col for col in service_cols if row[col] == 'Yes']), axis=1
)

# empty string means no services at all — replace with a readable label
df['bundle'] = df['bundle'].replace('', 'No Services')

bundle_analysis = df.groupby('bundle').agg(
    customer_count = ('customerID', 'count'), # how many customers share this exact bundle
    avg_monthly = ('MonthlyCharges', lambda x: round(x.mean(), 2)), # average monthly bill for this bundle
    churn_rate = ('Churn', lambda x: round((x == 'Yes').sum() / len(x) * 100, 2)),  # what % of them left
    avg_clv = ('TotalCharges', lambda x: round(x.mean(), 2)) # average lifetime value (total revenue per customer)
).reset_index()

# drop bundles with fewer than 30 customers — too small a sample to draw reliable conclusions
bundle_analysis = bundle_analysis[bundle_analysis['customer_count'] >= 30]

bundle_analysis.sort_values('avg_clv', ascending=False).head(10)